# The Resilient Distributed Dataset (RDD)




## 1. SparkContext

A `SparkConf` holds the app settings. A `SparkContext` (`sc`) is the handle to Spark.

- **appName:** shown in the Spark UI and logs
- **master:** where to run. `local` = this process (one thread). The Docker cluster is a different master (`spark://spark-master:7077`); we do not use it here.


In [ ]:
from pyspark import SparkConf, SparkContext

conf = SparkConf().setAppName("rdd-lesson").setMaster("local")
sc = SparkContext(conf=conf)

## 2. Create an RDD

`parallelize` copies a Python list into an RDD.


In [ ]:
data = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
dataDist = sc.parallelize(data)
type(dataDist)

## 3. Actions

Transformations are lazy. **Actions** run the job and return a Python value.

`collect()` pulls **every** element to the driver. Fine for 10 numbers. Never do this on the full CDC file.


In [ ]:
dataList = dataDist.collect()
print(type(dataList))
print(dataList)

`take(n)` returns the first *n* elements. Safe on large data.


In [ ]:
print(dataDist.take(3))
print(dataDist.takeOrdered(2))
print(dataDist.takeSample(False, 3))  # False = without replacement

`count`, `countByValue`, `top`, `max`, `min`.


In [ ]:
print(dataDist.count())
print(dict(dataDist.countByValue()))
print(dataDist.top(5))
print(dataDist.max(), dataDist.min())

## 4. Transformations: `map` and `reduce`

**Map** applies a function to each element. **Reduce** folds two values at a time into one. Both can take a `def` or a `lambda`.


In [ ]:
def compute_pow(d):
    return d * d

powDist = dataDist.map(compute_pow)
powDist.collect()

In [ ]:
powDist = dataDist.map(lambda d: d * d)
powDist.collect()

In [ ]:
def add(a, b):
    return a + b

print(dataDist.reduce(add))
print(dataDist.reduce(lambda a, b: a + b))

### `filter`

Keeps elements for which the function is true.


In [ ]:
words = [
    "Artificial Intelligence",
    "Machine Learning",
    "Reinforcement Learning",
    "Deep Learning",
    "Computer Vision",
    "Natural Language Processing",
    "Augmented Reality",
    "Blockchain",
    "Robotic",
    "Cyber Security",
]
wordsDist = sc.parallelize(words)
print(wordsDist.filter(lambda w: len(w) > 15))
print(wordsDist.filter(lambda w: len(w) > 15).collect())
print(wordsDist.filter(lambda w: w[0].lower() in "aeiou").collect())

## 5. Combining RDDs

`union` concatenates. `subtract` keeps values in the first RDD that are not in the second.


In [ ]:
dist1 = sc.parallelize([1, 2, 3, 4, 5])
dist2 = sc.parallelize([5, 6, 7, 8, 9])

print("union:", dist1.union(dist2).collect())
print("subtract:", dist1.subtract(dist2).collect())

### Wide transformations

These need data from more than one partition (a shuffle). Partitions themselves: next session.

`intersection` = in both. `distinct` = unique values. `cartesian` = all pairs.


In [ ]:
print("intersection:", dist1.intersection(dist2).collect())
print("cartesian:", dist1.cartesian(dist2).collect())

namesDist = sc.parallelize(["Giuseppe", "Francesco", "Antonio", "Antonio", "Giuseppe"])
print("distinct:", namesDist.distinct().collect())

### Key–value: `sortByKey` and `join`

`join` returns `(k, (v1, v2))` for matching keys.


In [ ]:
pairRDD = sc.parallelize([(1, 5), (1, 10), (2, 4), (3, 1), (2, 6)])
print(pairRDD.sortByKey().collect())

rdd1 = sc.parallelize([("a", 1), ("b", 4)])
rdd2 = sc.parallelize([("a", 2), ("a", 3)])
print(sorted(rdd1.join(rdd2).collect()))

# 6. Real data: CDC mortality 2021

CDC / NCHS *Multiple Cause-of-Death* public-use file: one **fixed-width** line per US death certificate. No header, no commas. Fields sit at column positions ([record layout](https://www.cdc.gov/nchs/data/dvs/Multiple-Cause-Record_Layout_2021.pdf)).

**In class we use a ~40,000-line sample** (every 87th record, months still look like 2021). The full file is 3.47 million lines / 2.6 GB (`mort2021us.txt`). Same slices. Do not `collect()` the full file.

| field | columns (1-based) | Python slice | codes |
| --- | --- | --- | --- |
| month of death | 65–66 | `[64:66]` | `01`–`12` |
| sex | 69 | `[68]` | `M` / `F` |
| age (years) | 71–73 | `[70:73]` | 999 = not stated |
| underlying cause | 146–149 | `[145:149]` | ICD-10, e.g. `U071` COVID-19 |


In [ ]:
from pathlib import Path

here = Path.cwd()
candidates = [
    Path("/opt/spark/data/cdc/mort2021us_sample.txt"),
    here / "cdc" / "mort2021us_sample.txt",
    here.parent / "book_data" / "cdc" / "mort2021us_sample.txt",
    here / "book_data" / "cdc" / "mort2021us_sample.txt",
]
CDC_PATH = str(next(p for p in candidates if p.exists()))
print(CDC_PATH)

lines = sc.textFile(CDC_PATH)
print(type(lines))
lines.take(2)

Count the lines. Instant on the sample; a real job on the 2.6 GB file.


In [ ]:
lines.count()

### Parse with `map`

`map` is lazy. `take` / `count` run the job.


In [ ]:
def parse(row):
    return {
        "month": row[64:66],
        "sex": row[68],
        "age": int(row[70:73]),
        "cause": row[145:149].strip(),
    }

deaths = lines.map(parse)
deaths.take(5)

### `filter`
Women, then COVID-19 (`U071`).


In [ ]:
women = deaths.filter(lambda d: d["sex"] == "F")
print("women (sample):", women.count())
women.take(3)

In [ ]:
covid = deaths.filter(lambda d: d["cause"] == "U071")
print("COVID-19 (sample):", covid.count())
covid.take(3)

### `countByValue`
Same action as on `[0, 1, …, 9]`.


In [ ]:
dict(deaths.map(lambda d: d["sex"]).countByValue())

In [ ]:
by_month = deaths.map(lambda d: d["month"]).countByValue()
sorted(by_month.items())

### `map` + `reduce`
COVID count: map each record to `1` or `0`, then add.


In [ ]:
covid_n = deaths.map(lambda d: 1 if d["cause"] == "U071" else 0).reduce(lambda a, b: a + b)
print(covid_n)

Top causes: map to the ICD code, `countByValue`, sort.


In [ ]:
causes = deaths.map(lambda d: d["cause"]).countByValue()
sorted(causes.items(), key=lambda kv: -kv[1])[:10]

### Narrow vs wide: time the difference

On 40k rows both finish in a blink. Copy each record **15 times** (~600k) and **cache**, so we measure CPU + shuffle, not the text-file read.

- **Narrow** (`filter`, `map`): each record stays in its partition. One stage.
- **Wide** (`join` on a unique id): Spark must **shuffle** both sides so matching keys land together. Same *N* output rows — the extra time is the shuffle, not a combinatorial explosion.

`groupByKey` on cause is a bad demo here: only ~1,300 keys, so the gap is tiny. A 1-to-1 `join` makes the shuffle obvious. Watch [Spark UI](http://localhost:4040): narrow is one stage; the join has shuffle write / shuffle read.

In [ ]:
import time

repeated = deaths.flatMap(lambda d: [d] * 15).cache()
print("cached rows:", repeated.count())


def timed(label, fn):
    t0 = time.perf_counter()
    result = fn()
    print(f"{label}: {time.perf_counter() - t0:.2f}s  -> {result}")


def run_narrow():
    return (
        repeated.filter(lambda d: d["age"] != 999)
        .map(lambda d: d["age"] * d["age"] + hash(d["cause"]) % 97)
        .filter(lambda score: score > 0)
        .count()
    )


def run_wide():
    left = repeated.zipWithIndex().map(lambda x: (x[1], x[0]))
    right = left.mapValues(lambda d: d["cause"])
    return left.join(right).count()


run_narrow()  # warmup so the first timed run is not the cache fill
timed("narrow  filter + map + count", run_narrow)
timed("wide    join on unique id", run_wide)

In [ ]:
sc.stop()